# Interactive Charts : Energy Storage Analytics

This notebook rebuilds the key visualisations from Notebooks 1 and 2 
using Plotly ; an interactive charting library that allows zooming, 
hovering, and clicking on data points directly in the notebook.

**Charts included:**

1. Total energy capacity by bus
2. Efficiency vs energy capacity scatter plot
3. Average efficiency by bus with storage unit count
4. Duration distribution

---


In [1]:
import pandas as pd
import plotly.express as px

# Load and prepare data
df1 = pd.read_excel("../data/stordata_capacity_vlookup.xlsx",
                    sheet_name="Sheet1",
                    usecols=[0,1,2,3])

df2 = pd.read_excel("../data/stordata_capacity_vlookup.xlsx",
                    sheet_name="Sheet2",
                    usecols=[1,2])

df2.set_index("id", inplace=True)
df1["power capability"] = df1["id"].map(df2["power capability"])
df1["energy capacity (kWh)"] = df1["power capability"] * df1["duration"]
df1["bus"] = df1["bus"].str.replace("bus 2", "bus_2")

df1.head()

,id,bus,Efficiency,duration,power capability,energy capacity (kWh)
0,stor1,bus_3,0.76,4.0,8.0,32.0
1,stor2,bus_3,0.87,1.0,5.0,5.0
2,stor3,bus_3,0.85,0.5,18.0,9.0
3,stor4,bus_2,0.89,4.0,12.0,48.0
4,stor5,bus_2,0.87,3.0,5.0,15.0


### Chart 1 : Total Energy Capacity by Bus

An interactive bar chart showing total energy capacity per bus.
Hover over each bar to see exact values.

---


In [10]:
# data PREPARATION
total_by_bus = df1.groupby("bus").agg(
    total_capacity=("energy capacity (kWh)", "sum"),
    number_of_units=("id", "count")
).reset_index()

total_by_bus.columns = ["Bus", "Total Energy Capacity (kWh)", "Number of Storage Units"]

# Interactive bar chart
fig = px.bar(
    total_by_bus,
    x="Bus",
    y="Total Energy Capacity (kWh)",
    color="Bus",
    hover_data=["Number of Storage Units"],
    title="Total Energy Capacity by Bus"
)

fig.show()

<span style="color:grey;">*Figure 1 : Total energy capacity per bus across all 25 storage units. 
bus_1 dominates with 1,099 kWh ; hover over each bar to see the 
number of storage units contributing to each total.*</span>

---

### Chart 2 : Efficiency vs Energy Capacity

An interactive scatter plot showing the relationship between 
efficiency and energy capacity for all 25 storage units.
Hover over each dot to see the full details of that storage unit.

---

In [5]:
# Interactive scatter plot
fig = px.scatter(
    df1,
    x="Efficiency",
    y="energy capacity (kWh)",
    color="bus",
    hover_data=["id", "duration", "power capability"],
    title="Efficiency vs Energy Capacity by Bus for All 25 Storage Units",
    labels={
        "energy capacity (kWh)": "Energy Capacity (kWh)",
        "Efficiency": "Efficiency"
    }
)

fig.show()

<span style="color:grey;">*Figure 2 : Each dot represents one storage unit plotted by efficiency 
and energy capacity. Dots are coloured by bus. Hover over any dot 
to see the full profile of that storage unit including id, duration 
and power capability. Storage units in the bottom right quadrant 
; high capacity but low efficiency ; are the most concerning.*</span>

### Chart 3 : Average Efficiency by Bus with Storage Unit Count

An interactive bar chart showing average efficiency per bus.
The number of storage units per bus is shown on hover ; 
providing context for whether the average is meaningful or 
based on a small sample.

---

In [6]:
# Prepare data
avg_efficiency = df1.groupby("bus").agg(
    average_efficiency=("Efficiency", "mean"),
    number_of_units=("id", "count")
).reset_index()

avg_efficiency.columns = ["Bus", "Average Efficiency", "Number of Storage Units"]

# Interactive bar chart
fig = px.bar(
    avg_efficiency,
    x="Bus",
    y="Average Efficiency",
    color="Bus",
    hover_data=["Number of Storage Units"],
    title="Average Efficiency by Bus",
    labels={"Average Efficiency": "Average Efficiency"}
)

fig.update_yaxes(range=[0.7, 0.95])
fig.show()

<span style="color:grey;">*Figure 3 : Average efficiency per bus. bus_4 appears highest but 
hover data reveals it contains only 1 storage unit ; making its 
average less statistically meaningful than buses with larger samples. 
The y-axis starts at 0.70 to make differences between buses visible.*</span>

### Chart 4 : Duration Distribution

An interactive bar chart showing how many storage units 
share each duration value ; revealing the most common 
duration configurations in this grid.

---

In [7]:
# Prepare data
duration_counts = df1["duration"].value_counts().reset_index()
duration_counts.columns = ["Duration (hours)", "Number of Storage Units"]
duration_counts = duration_counts.sort_values("Duration (hours)")

# Interactive bar chart
fig = px.bar(
    duration_counts,
    x="Duration (hours)",
    y="Number of Storage Units",
    color="Duration (hours)",
    title="Distribution of Storage Unit Durations",
    labels={
        "Duration (hours)": "Duration (hours)",
        "Number of Storage Units": "Number of Storage Units"
    }
)

fig.show()

<span style="color:grey;">*Figure 4 : Duration distribution across all 25 storage units. 
4 hours is the dominant configuration with 9 storage units ; 
suggesting this represents the design standard for this grid.*</span>

## Conclusions

This notebook rebuilt the key visualisations from Notebooks 1 and 2 
using Plotly ; transforming static matplotlib charts into fully 
interactive visualisations.

**Key advantages of Plotly over matplotlib:**

- Hover tooltips reveal full storage unit details without cluttering 
  the chart
- Legend interactivity allows isolation of individual buses for 
  focused comparison
- Zoom and pan functionality allows deeper exploration of dense 
  data regions
- Charts are significantly more impressive in client presentations 
  and stakeholder meetings

**Key visual insights confirmed:**

- bus_1 dominates raw total energy capacity but drops significantly 
  when efficiency is used as a filter
- stor24 and stor19 are clear outliers ; high capacity but low 
  efficiency ; visible immediately on the scatter plot
- 4 hours is the dominant duration configuration ; confirmed visually 
  by the distribution chart
- bus_4 appears highly efficient but hover data reveals it contains 
  only 1 storage unit ; context that a static chart would miss

## Next Steps
- Notebook 4 : Cost-effectiveness scoring ; combining efficiency, 
  capacity and estimated cost to rank storage units by value